# Review Streaming (Kafka) - Functionized

Each function is defined in its own cell for easy conversion into a PySpark job later.


## Imports and shared Spark types


In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


## get_env: read environment variables with defaults


In [ ]:
def get_env(name: str, default: str) -> str:
    """Return the environment variable value or a default."""
    return os.getenv(name, default)


## build_spark: create a SparkSession


In [ ]:
def build_spark(app_name: str = 'review-streaming') -> SparkSession:
    """Create and return a SparkSession with the given app name."""
    return SparkSession.builder.appName(app_name).getOrCreate()


## review_schema: schema for review JSON payloads


In [ ]:
def review_schema() -> StructType:
    """Return a StructType describing the review JSON payload."""
    return StructType([
        StructField('review_id', StringType(), True),
        StructField('user_id', StringType(), True),
        StructField('business_id', StringType(), True),
        StructField('stars', IntegerType(), True),
        StructField('useful', IntegerType(), True),
        StructField('funny', IntegerType(), True),
        StructField('cool', IntegerType(), True),
        StructField('text', StringType(), True),
        StructField('date', StringType(), True),
    ])


## build_reviews_stream: read Kafka and parse JSON


In [ ]:
def build_reviews_stream(spark: SparkSession):
    """Read Kafka and parse JSON payloads into a streaming DataFrame."""
    kafka_bootstrap = get_env('KAFKA_BOOTSTRAP_SERVERS', 'broker:29092')
    kafka_topic = get_env('KAFKA_TOPIC_REVIEW', 'raw_data_review')

    kafka_df = (
        spark.readStream.format('kafka')
        .option('kafka.bootstrap.servers', kafka_bootstrap)
        .option('subscribe', kafka_topic)
        .option('startingOffsets', 'latest')
        .load()
    )

    schema = review_schema()
    return (
        kafka_df.selectExpr('CAST(value AS STRING) AS json_str')
        .select(from_json(col('json_str'), schema).alias('review'))
        .select('review.*')
    )


## start_memory_sink: write the stream to memory


In [ ]:
def start_memory_sink(reviews_stream, query_name: str = 'reviews_stream'):
    """Write the stream to an in-memory table for ad-hoc inspection."""
    return (
        reviews_stream.writeStream
        .format('memory')
        .queryName(query_name)
        .outputMode('append')
        .start()
    )


## show_streamed: view rows from the memory table


In [ ]:
def show_streamed(spark: SparkSession, limit: int = 20):
    """Display the most recent rows captured by the memory sink."""
    df = spark.table('reviews_stream')
    df.orderBy(df['date'].desc()).show(limit, truncate=False)


## show_streamed_count: view counts of rows from the memory table

In [20]:
def show_streamed_count(spark: SparkSession):
    """Display the number of rows captured in the memory sink."""
    count = spark.table("reviews_stream").count()
    print(f"reviews_stream count: {count}")


## stop_all_streams: stop active streaming queries


In [ ]:
def stop_all_streams(spark: SparkSession):
    """Stop every active streaming query attached to this Spark session."""
    for stream in spark.streams.active:
        stream.stop()


## Manual run (notebook)
Run these cells in order.


In [13]:
spark = build_spark()
spark


In [14]:
reviews_stream = build_reviews_stream(spark)
query = start_memory_sink(reviews_stream)
query

26/01/05 04:59:11 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-ec91392b-6914-4ca9-a93e-27cf0ad69b85. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/01/05 04:59:11 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [ ]:
# Re-run to refresh
show_streamed(spark, limit=20)


In [21]:
# Re-run to refresh
show_streamed_count(spark)


reviews_stream count: 127


In [ ]:
# Stop all running streams
stop_all_streams(spark)
